# 04 Mel 频谱图、CQT、MFCC 与 Chroma

内容脉络：
1. **Mel 频谱图**：按具有心理声学动机的频带重新组织频率信息
2. **CQT（常数 Q 变换）**：按音乐音高结构（对数频率）组织
3. **MFCC**：对 Mel 谱做 DCT 压缩，得到紧凑系数
4. **Chroma**：八度折叠到 12 维音级向量

这些表示采用不同方式组织频率信息：Mel 与 MFCC 通常由 STFT 功率谱派生，CQT 直接使用对数间隔滤波器，Chroma 则可由 STFT 或 CQT 折叠得到。

## 1. 环境自检与配置

In [ ]:
import sys
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from pathlib import Path

import librosa
import librosa.display

SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_FIG_DIR.mkdir(exist_ok=True)

print(f"librosa={librosa.__version__}")

## 2. Mel 频谱图

Mel 滤波器组把 STFT 功率谱聚合为较少的感知频带。常见计算链路：
1. STFT 功率谱
2. Mel 三角滤波器组加权求和
3. 对 mel 功率取对数或 dB

Mel 尺度描述频率/音高感知的近似关系，不是响度模型；`n_mels=128` 是音乐任务常见起点之一，并非普遍最优值。


In [ ]:
# 加载钢琴片段
piano_samples, sr = librosa.load(
    DATASET_DIR / "piano_solo.wav", sr=SAMPLE_RATE, mono=True, offset=10.0, duration=3.0
)

stft = librosa.stft(piano_samples, n_fft=2048, hop_length=512, window="hann")
linear_logspec = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

mel_spec = librosa.feature.melspectrogram(
    y=piano_samples, sr=SAMPLE_RATE, n_fft=2048,
    hop_length=512, n_mels=128, power=2.0
)
mel_logspec = librosa.power_to_db(mel_spec, ref=np.max)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)

img0 = librosa.display.specshow(
    linear_logspec, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="linear", ax=axes[0], cmap="Greys_r"
)
axes[0].set_title("STFT（线性 Hz 频率轴）")
axes[0].set_xlabel("时间 (s)")
axes[0].set_ylim(0, 8000)
fig.colorbar(img0, ax=axes[0], format="%+2.0f dB")

img1 = librosa.display.specshow(
    mel_logspec, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="mel", ax=axes[1], cmap="Greys_r"
)
axes[1].set_title("Mel 频谱图（n_mels=128）")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylim(0, 8000)
fig.colorbar(img1, ax=axes[1], format="%+2.0f dB")

plt.suptitle("线性频率轴 vs Mel 频带：钢琴独奏", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "mel_vs_linear.png", dpi=600, bbox_inches="tight")
plt.show()


### Mel 滤波器组的形状

`librosa.filters.mel` 返回的矩阵形状为 `(n_mels, 1 + floor(n_fft/2))`。每一行是一个三角滤波器，对 STFT 的频带做加权。低频区滤波器较窄、排列较密，高频区较宽、排列较疏；这描述的是 mel 频带划分，不等同于原始 STFT 的实际频率分辨能力。

In [ ]:
n_mels_demo = 40  # 用较少通道便于可视化
mel_filterbank = librosa.filters.mel(sr=SAMPLE_RATE, n_fft=2048, n_mels=n_mels_demo)
freq_axis = np.fft.rfftfreq(2048, 1 / SAMPLE_RATE)

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(n_mels_demo):
    ax.plot(freq_axis, mel_filterbank[i], alpha=0.7, lw=0.8)
ax.set_xlim(0, 8000)
ax.set_xlabel("频率 (Hz)")
ax.set_ylabel("滤波器权重")
ax.set_title(f"Mel 滤波器组（n_mels={n_mels_demo}, n_fft=2048）")
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "mel_filterbank.png", dpi=600, bbox_inches="tight")
plt.show()

## 3. 常数 Q 变换（CQT）

CQT 用对数间隔的中心频率和近似恒定的 $Q=f/\Delta f$ 组织滤波器。低频滤波器需要更长的有效窗，因此**低频时间分辨率较差**；高频滤波器较短，时间定位相对更细。

最高中心频率低于 Nyquist 只是一项粗略检查，完整滤波器通带也必须可实现。`hop_length` 取 2 的幂便于 librosa 的递归下采样整除，但不是 CQT 正确性的普遍硬条件。


In [ ]:
# 用钢琴片段对比 STFT 与 CQT
# CQT 参数：每个八度 36 bins（即每半音 3 bins），从 C1 起覆盖 7 个完整八度（最高中心频率低于 C8）
bins_per_octave = 36
fmin_cqt = librosa.note_to_hz("C1")
n_bins = 7 * bins_per_octave

stft = librosa.stft(piano_samples, n_fft=8192, hop_length=512, window="hann")
stft_log = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

cqt_complex = librosa.cqt(piano_samples, sr=SAMPLE_RATE, hop_length=512,
                          fmin=fmin_cqt, n_bins=n_bins,
                          bins_per_octave=bins_per_octave)
cqt_log = librosa.amplitude_to_db(np.abs(cqt_complex), ref=np.max)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

img0 = librosa.display.specshow(
    stft_log, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="hz", ax=axes[0], cmap="Greys_r"
)
axes[0].set_title("STFT（n_fft=8192，线性频率）")
axes[0].set_xlabel("时间 (s)")
axes[0].set_ylim(50, SAMPLE_RATE // 2)
fig.colorbar(img0, ax=axes[0], format="%+2.0f dB")

img1 = librosa.display.specshow(
    cqt_log, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="cqt_hz", fmin=fmin_cqt,
    bins_per_octave=bins_per_octave, ax=axes[1], cmap="Greys_r"
)
axes[1].set_title(f"CQT（bins_per_octave={bins_per_octave}）")
axes[1].set_xlabel("时间 (s)")
fig.colorbar(img1, ax=axes[1], format="%+2.0f dB")

plt.suptitle("STFT 与 CQT：钢琴独奏的低频细节", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "stft_vs_cqt.png", dpi=600, bbox_inches="tight")
plt.show()

**观察**：
- STFT 数据本身按等 Hz bin 采样；这里用对数显示轴只是改变可视坐标
- CQT 的滤波器中心频率本身按对数排列；每八度 36 bins 时，每半音有 3 bins
- 图像差异只说明当前片段与参数下的表示效果，不能把某一种表示视为所有音高任务的固定最优输入


### HCQT：谐波常数 Q 变换（Harmonic CQT）

HCQT 把若干谐波倍率对应的 CQT 视图堆成通道，使同一候选基频及其谐波证据在通道维对齐。

下面按每个谐波倍率直接计算一张 CQT：第 $h$ 个通道的起始频率设为 $h f_{\min}$，使同一行对应候选基频 $f$ 及其 $hf$ 处的谱量。这样无需把非整数 bin 位移四舍五入。各层最终堆叠成 $(\text{bins},\text{frames},\text{harmonics})$。

这是一种对齐谐波证据的表示，不保证消除谐波重叠或直接给出正确 F0。


In [ ]:
# HCQT：谐波常数 Q 变换可视化

# 参数设置
fmin = librosa.note_to_hz('C1')          # 最低基频 ~32.7 Hz
bins_per_octave = 36                     # 每个八度 36 bins（每半音 3 bins）
n_bins_base = 5 * bins_per_octave        # 基频层从 C1 起覆盖 5 个八度（最高中心频率低于 C6）
harmonics = [1, 2, 3, 4, 5, 6]           # 提取前 6 个谐波层

# 为每个谐波倍率直接计算一张 CQT；同一行共享候选基频轴
hcqt_layers = []
for h in harmonics:
    layer = np.abs(librosa.cqt(
        piano_samples, sr=SAMPLE_RATE, hop_length=512,
        fmin=h * fmin, n_bins=n_bins_base,
        bins_per_octave=bins_per_octave,
    ))
    hcqt_layers.append(layer)

assert len({layer.shape for layer in hcqt_layers}) == 1
hcqt = np.stack(hcqt_layers, axis=-1)
assert hcqt.shape == (n_bins_base, hcqt_layers[0].shape[1], len(harmonics))
hcqt_ref = max(float(np.max(layer)) for layer in hcqt_layers)

# 可视化各谐波层（2×3 布局，对称填满）
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, (h, layer) in enumerate(zip(harmonics, hcqt_layers)):
    layer_db = librosa.amplitude_to_db(layer, ref=hcqt_ref)
    img = librosa.display.specshow(
        layer_db, sr=SAMPLE_RATE, hop_length=512,
        fmin=fmin, bins_per_octave=bins_per_octave,
        x_axis='time', y_axis='cqt_hz', ax=axes[i], cmap='Greys_r'
    )
    axes[i].set_title(f'谐波通道 h={h}：读取 {h}f₀ 附近谱量')
    axes[i].set_xlabel('时间 (s)')
    axes[i].set_ylabel('候选基频 (Hz)')
    fig.colorbar(img, ax=axes[i], format='%+2.0f dB')

plt.suptitle('HCQT：按谐波倍数分别计算并对齐通道', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / 'hcqt_harmonics.png', dpi=600, bbox_inches='tight')
plt.show()


## 4. MFCC：从 log-mel 到紧凑系数

MFCC 的典型链路是 mel 功率谱、对数压缩与 DCT。DCT 归一化、mel 公式、参考值和是否保留 $c_0$ 都会改变数值。$c_0$ 反映平均 log-mel 水平，通常与帧能量相关，但不是严格的物理能量或标准响度。


In [ ]:
# 同一前端的 log-mel、20 维 MFCC 与一阶差分
mel_power_mfcc = librosa.feature.melspectrogram(
    y=piano_samples, sr=SAMPLE_RATE, n_fft=2048,
    hop_length=512, n_mels=128, power=2.0
)
log_mel_mfcc = librosa.power_to_db(mel_power_mfcc, ref=np.max)
mfcc_20 = librosa.feature.mfcc(S=log_mel_mfcc, n_mfcc=20, dct_type=2, norm="ortho")
delta_mfcc = librosa.feature.delta(mfcc_20)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)

img0 = librosa.display.specshow(
    log_mel_mfcc, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="mel", ax=axes[0], cmap="Greys_r"
)
axes[0].set_title("log-mel（128 个频带）")
axes[0].set_xlabel("时间 (s)")
fig.colorbar(img0, ax=axes[0], format="%+2.0f dB")

img1 = librosa.display.specshow(
    mfcc_20, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", ax=axes[1], cmap="Greys_r"
)
axes[1].set_title("MFCC（20 个系数）")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylabel("MFCC 序号")
fig.colorbar(img1, ax=axes[1])

img2 = librosa.display.specshow(
    delta_mfcc, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", ax=axes[2], cmap="Greys_r"
)
axes[2].set_title("MFCC 一阶差分")
axes[2].set_xlabel("时间 (s)")
axes[2].set_ylabel("MFCC 序号")
fig.colorbar(img2, ax=axes[2])

plt.suptitle("同一前端：log-mel、MFCC 与一阶差分", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "mfcc_comparison.png", dpi=600, bbox_inches="tight")
plt.show()


**注意**：$c_0$ 是各 mel 通道 log 值的平均分量（随 DCT 归一化相差常数），通常与能量相关，但不能直接当作响度。低阶系数主要刻画沿频带方向缓慢变化的谱包络；需要多少系数应由任务和验证数据决定。


## 5. Chroma：八度折叠后的音级向量

**动机**：在许多和弦与调性任务中，C3 和 C4 共享 C 这一音级身份。Chroma 把谱量折叠到 12 个音级，每帧输出一个 12 维向量；代价是丢失音区、低音/转位线索和声部配置，因此不能说八度位置在所有和声问题中都不重要。

两种计算方式：
- `chroma_stft`：基于 STFT 谱折叠；已有 STFT、计算资源受限或需要直接控制线性谱前端时可作为起点
- `chroma_cqt`：基于 CQT 折叠；当对数音高网格与低频音级分辨率更契合任务时可采用

二者没有脱离数据、窗长、调音与后处理的固定优劣关系。

In [ ]:
# 用人声歌曲演示 Chroma（旋律变化丰富）
vox_samples, sr = librosa.load(
    DATASET_DIR / "xiaohetang_vox.wav", sr=SAMPLE_RATE, mono=True, offset=30.0, duration=5.0
)

chroma_stft = librosa.feature.chroma_stft(y=vox_samples, sr=SAMPLE_RATE, hop_length=512)
chroma_cqt = librosa.feature.chroma_cqt(y=vox_samples, sr=SAMPLE_RATE, hop_length=512)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

img0 = librosa.display.specshow(
    chroma_stft, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="chroma", ax=axes[0], cmap="Greys_r"
)
axes[0].set_title("基于 STFT 的 chroma")
axes[0].set_xlabel("")
axes[0].set_ylabel("音级")
fig.colorbar(img0, ax=axes[0])

img1 = librosa.display.specshow(
    chroma_cqt, sr=SAMPLE_RATE, hop_length=512,
    x_axis="time", y_axis="chroma", ax=axes[1], cmap="Greys_r"
)
axes[1].set_title("基于 CQT 的 chroma")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylabel("音级")
fig.colorbar(img1, ax=axes[1])

plt.suptitle("chroma：12 维音级向量", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "chroma_comparison.png", dpi=600, bbox_inches="tight")
plt.show()

**观察**：
- 在当前片段与参数下，基于 CQT 的 chroma 部分音级轨迹更集中；这不是跨数据的普遍性能结论
- 基于 STFT 的 chroma 低频音级分配受 FFT 窗长、bin 间距与调音偏差影响；增大窗长或改变前端会改变对比
> Chroma 是后续和弦识别（`07` Notebook 模板法）的核心输入之一。

## 6. 四种表示并排总览

同一段钢琴音频的 STFT、Mel、CQT 与 Chroma。四者分别突出物理频率、感知频带、对数音高与八度折叠；MFCC 已在上一节单独展示。


In [ ]:
# 重新计算四种表示，统一时间步长
stft = librosa.stft(piano_samples, n_fft=2048, hop_length=512)
linear = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

mel = librosa.power_to_db(
    librosa.feature.melspectrogram(
        y=piano_samples, sr=SAMPLE_RATE, n_fft=2048,
        hop_length=512, n_mels=128, power=2.0
    ), ref=np.max
)

cqt = librosa.amplitude_to_db(
    np.abs(librosa.cqt(
        piano_samples, sr=SAMPLE_RATE, hop_length=512,
        fmin=librosa.note_to_hz("C1"), n_bins=84, bins_per_octave=12
    )), ref=np.max
)

chroma = librosa.feature.chroma_cqt(y=piano_samples, sr=SAMPLE_RATE, hop_length=512)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
representations = [
    ("STFT（线性 Hz）", linear, "linear"),
    ("Mel（128 个频带）", mel, "mel"),
    ("CQT（C1–B7）", cqt, "cqt_hz"),
    ("Chroma（12 个音级）", chroma, "chroma"),
]
y_labels = {
    "linear": "频率 (Hz)",
    "mel": "Mel 频率 (Hz)",
    "cqt_hz": "频率 (Hz)",
    "chroma": "音级",
}

for ax, (title, data, y_axis) in zip(axes.flatten(), representations):
    img = librosa.display.specshow(
        data, sr=SAMPLE_RATE, hop_length=512,
        x_axis="time", y_axis=y_axis, ax=ax, cmap="Greys_r"
    )
    ax.set_title(title)
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel(y_labels[y_axis])
    fig.colorbar(img, ax=ax)

plt.suptitle("同一段钢琴的四种表示", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "four_representations_overview.png", dpi=600, bbox_inches="tight")
plt.show()


## 7. 小结

本 Notebook 展示了五类常见表示，其中四类进入总览图：

| 表示 | 组织方式 | 主要保留的信息 |
|:---|:---|:---|
| STFT | 等 Hz DFT bin | 物理谱形与相位入口 |
| Mel | mel 滤波器组 | 压缩后的感知频带能量 |
| CQT | 对数中心频率 | 音高间隔结构 |
| MFCC | log-mel 后做 DCT | 低维谱包络系数 |
| Chroma | 八度折叠 | 12 个音级的相对分布 |

表示选择取决于任务；CQT 网格更密不等于自动支持某种非西方调律，Chroma 也会主动丢弃音区与细粒度频率信息。


In [ ]:
print("本 Notebook 生成的图像文件：")
for prefix in ["mel_", "stft_vs_cqt", "hcqt_", "mfcc_", "chroma_", "four_rep"]:
    for f in OUTPUT_FIG_DIR.glob(f"{prefix}*.png"):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s} {size_kb:8.1f} KB")